In [1]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [8]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_bm25,
    full_dense,
    full_hybrid,
    chunk_dense
)
import sys
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
nest_asyncio.apply()

In [9]:
REPO_ROOT     = Path("/home/ubuntu/work/dahyeon")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data.json"

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

In [10]:
def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


In [11]:
# ============================================================
# Strategy0/1/2 Dense Retrieval 비교
# - embedding model: KURE 고정
# - strategy0, strategy2: full_dense
# - strategy1: chunk_dense
# - isbn만 저장
# ============================================================

from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm

retrieval_configs = {
        "strategy0": {
        "func": full_dense,
        "index_name": "books_review_full",
        "type": "full"
    },

    "strategy1": {
        "func": chunk_dense,
        "index_name": "books_chunk_strategy1",
        "type": "chunk"
    },
    "strategy2": {
        "func": full_dense,
        "index_name": "books_chunk_strategy2",
        "type": "full"
    }
}

EMBEDDING_MODEL = "kure"
EMBEDDING_FIELD = "embedding_kure"

CANDIDATE_SIZE = 100
TOP_K = 20

all_candidates = []

for scenario in tqdm(scenarios, desc="strategy retrieval"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    for strategy_name, config in retrieval_configs.items():

        retrieval_func = config["func"]
        index_name = config["index_name"]
        retrieval_type = config["type"]

        size = CANDIDATE_SIZE if retrieval_type == "chunk" else TOP_K

        re_dense = retrieval_func(
            sample_result,
            size=size,
            index_name=index_name,
            embedding_field=EMBEDDING_FIELD,
            embedding_model=EMBEDDING_MODEL,
            small_category_embeddings=None
        )

        seen_isbns = set()
        rank = 1

        for book in re_dense:
            isbn = str(book["isbn"])

            if isbn in seen_isbns:
                continue

            seen_isbns.add(isbn)

            all_candidates.append({
                "query_id": query_id,
                "embedding_model": EMBEDDING_MODEL,
                "embedding_field": EMBEDDING_FIELD,

                "strategy": strategy_name,
                "retrieval_type": retrieval_type,
                "index_name": index_name,

                "rank": rank,
                "dense_score": book.get("dense_score"),
                "score": book.get("score"),

                "isbn": isbn,
                "title": book.get("title"),
                "author": book.get("author"),
                "publisher": book.get("publisher"),
                "publish_date": book.get("publish_date"),
                "page": book.get("page"),

                "large_cate": book.get("large_cate"),
                "mid_cate": book.get("mid_cate"),
                "small_cate": book.get("small_cate"),

                "book_intro": book.get("book_intro"),
                "book_index": book.get("book_index"),
                "book_index_text": book.get("book_index_text"),

                "review": book.get("review"),
                "review_text": book.get("review_text"),

                "chunk_type": book.get("chunk_type"),
                "chunk_text": book.get("chunk_text"),
            })

            rank += 1

            if rank > TOP_K:
                break

print(f"\n전체 후보 도서: {len(all_candidates):,}")

strategy_compare_df = pd.DataFrame(all_candidates)
strategy_compare_df.head()

OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_strategy_dense_eval.jsonl")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for item in all_candidates:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"저장 완료: {OUTPUT_PATH}")
print(f"총 저장 개수: {len(all_candidates):,}")

strategy retrieval: 100%|██████████| 21/21 [02:37<00:00,  7.52s/it]


전체 후보 도서: 1,260
저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/chunk_strategy_dense_eval.jsonl
총 저장 개수: 1,260


In [12]:
# =====================================================
# 1. 파일 경로
# =====================================================

from pathlib import Path
import sys
import json
import pandas as pd

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_corrected.json")
DENSE_RESULT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_strategy_dense_eval.jsonl")

REPO_ROOT = Path("/home/ubuntu/work/dahyeon")
sys.path.insert(0, str(REPO_ROOT / "evaluation" / "metrics"))

from metrics import hit_at_k, recall_at_k, mrr_at_k, ndcg_at_k

K_VALUES = [20]
RELEVANCE_THRESHOLD = 2

# =====================================================
# 2. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

print("goldset 개수:", len(gold_df))
display(gold_df.head())

# =====================================================
# 3. strategy dense retrieval 결과 로드
# =====================================================

dense_rows = []

with DENSE_RESULT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        dense_rows.append(json.loads(line))

dense_df = pd.DataFrame(dense_rows)

print("dense 결과 개수:", len(dense_df))
display(dense_df.head())

# =====================================================
# 4. query_id별 정답 ISBN 만들기
#    final_grade >= 2 를 relevant로 사용
# =====================================================

relevant_by_query = {}

for qid, g in gold_df.groupby("query_id"):
    relevant_isbns = (
        g[g["final_grade"] >= RELEVANCE_THRESHOLD]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    relevant_by_query[qid] = set(relevant_isbns)

print("query 수:", len(relevant_by_query))
print("relevant 있는 query 수:", sum(1 for v in relevant_by_query.values() if v))

# =====================================================
# 5. eval_data 구성
#    기준: strategy1 / strategy2 / strategy3
# =====================================================

eval_data = {}

for qid in relevant_by_query:
    eval_data[qid] = {
        "relevant_isbns": relevant_by_query[qid]
    }

for (qid, strategy), g in dense_df.groupby(["query_id", "strategy"]):
    ranked = (
        g.sort_values("rank")["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    if qid not in eval_data:
        eval_data[qid] = {
            "relevant_isbns": set()
        }

    eval_data[qid][strategy] = ranked

strategies = sorted(dense_df["strategy"].dropna().unique())

print("strategies:", strategies)

# =====================================================
# 6. strategy별 retrieval metric 계산
# =====================================================

def compute_retrieval_metrics(
    eval_data,
    retrievers,
    ks=K_VALUES,
):
    rows = []

    for qid, d in eval_data.items():
        rel = d["relevant_isbns"]

        if not rel:
            continue

        for src in retrievers:
            ranked = d.get(src, [])

            if not ranked:
                continue

            for k in ks:
                rows.append({
                    "query_id": qid,
                    "strategy": src,
                    "k": k,
                    "hit": hit_at_k(rel, ranked, k),
                    "recall": recall_at_k(rel, ranked, k),
                    "mrr": mrr_at_k(rel, ranked, k),
                    "ndcg": ndcg_at_k(rel, ranked, k),
                })

    return pd.DataFrame(rows)

retrieval_df = compute_retrieval_metrics(
    eval_data=eval_data,
    retrievers=strategies,
    ks=K_VALUES,
)

print(f"총 {len(retrieval_df)}개 행 (시나리오 × strategy × K)")
display(retrieval_df.head(9))

goldset 개수: 2519


,isbn,title,author,publisher,publish_date,page,large_cate,mid_cate,small_cate,book_intro,...,relevance_grade,binary_label,label_status,llm_raw_output,llm_error,final_grade,confidence,grade_j1,grade_j2,grade_j3
0,9772586200037,과학잡지 에피 (계간) : 13호 [2020] - 잡지 에피 13,편집부 저,이음,2020-09-02,260,"[자연/과학, 잡지]",[교양과학],[과학이야기],『에피』 13호 식량의 과학\n전세계 코로나 감염자가 2500만여 명에 육박하고 있...,...,0,0,success,"{\n ""relevance_grade"": 0,\n ""reason"": ""사용자가 ...",None,0,MEDIUM,0,0,1
1,9772586200037,과학잡지 에피 (계간) : 13호 [2020] - 잡지 에피 13,편집부 저,이음,2020-09-02,260,"[자연/과학, 잡지]",[교양과학],[과학이야기],『에피』 13호 식량의 과학\n전세계 코로나 감염자가 2500만여 명에 육박하고 있...,...,1,0,success,"{\n ""relevance_grade"": 1,\n ""reason"": ""교양과학 ...",None,1,HIGH,1,1,1
2,9788900035803,깊고 푸른밤 외,최인호 저,두산동아(단행),1995-05-01,473,[소설],[],[],19세기 말에서 지금 우리 시대에 이르는 근 백 년간의 소설 작품들을 국문학자 · ...,...,1,0,success,"{\n ""relevance_grade"": 1,\n ""reason"": ""사용자가 ...",None,1,HIGH,1,1,1
3,9788901046044,수다가 사람 살려,오한숙희 저,웅진지식하우스,2004-07-03,248,"[인문, 사회/정치]",[나라별 에세이],[한국에세이],"방송과 강의, 상담을 통해 수많은 사람들과 소통하는 가운데 수다가 가지고 있는 해소...",...,0,0,success,"{\n ""relevance_grade"": 0,\n ""reason"": ""사용자가 ...",None,0,MEDIUM,0,0,1
4,9788901053073,세상 청바지 - 정의로운 사회는 가능할까?,김창호 편,웅진지식하우스,2005-10-21,313,"[인문, 청소년]","[나라별 에세이, 철학]","[철학에세이, 한국에세이]",'스무 살을 위한 철학 청바지'는 인문학의 위기 속에서 대학생들의 철학교양을 높여 ...,...,1,0,success,"{\n ""relevance_grade"": 1,\n ""reason"": ""청소년 대...",None,1,HIGH,1,1,1


dense 결과 개수: 1260


,query_id,embedding_model,embedding_field,strategy,retrieval_type,index_name,rank,dense_score,score,isbn,...,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text,chunk_type,chunk_text
0,S01,kure,embedding_kure,strategy0,full,books_review_full,1,0.814955,0.814955,9791196539757,...,"[시/에세이, 인문]",[나라별 에세이],[한국에세이],"“인간존재와 삶의 의미는 무엇인지, 왜 사는지, 무엇을 위해 어떻게 살 것인지, 늙...",,,"{'recommended_for': '삶의 본질을 알고자 하거나 힘든 분들', 's...",속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드는 책입니다. 삶의 본질을 알고자 하거...,NaN,NaN
1,S01,kure,embedding_kure,strategy0,full,books_review_full,2,0.786085,0.786085,9791192942865,...,[시/에세이],[나라별 에세이],[한국에세이],"열여덟 소녀의 감성, 삶의 서사, 사랑의 방정식, 그리움과 설렘의 상관관계를 잔잔한...",,,None,,NaN,NaN
2,S01,kure,embedding_kure,strategy0,full,books_review_full,3,0.783358,0.783358,9791198387509,...,"[시/에세이, 인문]",[철학],"[교양철학, 철학에세이]","삶을 허툴지 않게 살아보겠다고 결심하는 순간, 우리는 누구나 철학자가 된다!!\n학...",,"초대의 글 철학의 위안 보이티우스 나, 이성인가, 욕망인가? 플라톤 뜨겁지만 냉정하...",None,,NaN,NaN
3,S01,kure,embedding_kure,strategy0,full,books_review_full,4,0.780992,0.780992,9791163383031,...,[시/에세이],[나라별 에세이],[한국에세이],"여자, 에세이를 만날 때\n에세이란, 일정 형식을 따르지 않고 인생이나 자연 또는 ...",,,"{'recommended_for': '', 'strengths': '글을 쓴다는 것...",책을 읽는 순간이자 자신의 이야기를 쓰고 싶은 마음을 담은 글들로 구성된 에세이집입...,NaN,NaN
4,S01,kure,embedding_kure,strategy0,full,books_review_full,5,0.779680,0.779680,9791157833733,...,"[시/에세이, 인문]","[나라별 에세이, 철학, 테마에세이]","[교양철학, 노년에세이, 한국에세이]","영원히 살고자 하는 욕망이 부른 인류세 시대,\n이제 우리는 나이 듦의 윤리를 말해...",,수명연장 과학 시대의 나이 듦과 삶의 의미 김종갑 노화가 질병이라는 말에 관하여 서...,None,,NaN,NaN


query 수: 21
relevant 있는 query 수: 21
strategies: ['strategy0', 'strategy1', 'strategy2']
총 63개 행 (시나리오 × strategy × K)


,query_id,strategy,k,hit,recall,mrr,ndcg
0,S01,strategy0,20,1,0.176471,1.000000,0.255664
1,S01,strategy1,20,1,0.411765,0.333333,0.343628
2,S01,strategy2,20,1,0.235294,1.000000,0.285118
3,S02,strategy0,20,0,0.000000,0.000000,0.000000
4,S02,strategy1,20,1,0.111111,0.062500,0.057504
5,S02,strategy2,20,0,0.000000,0.000000,0.000000
6,S03,strategy0,20,1,0.600000,1.000000,0.722727
7,S03,strategy1,20,1,0.800000,1.000000,0.732025
8,S03,strategy2,20,1,0.600000,1.000000,0.699215


In [18]:
def compute_retrieval_metrics(
    eval_data,
    retrievers,
    ks=K_VALUES,
):
    rows = []

    for qid, d in eval_data.items():
        rel = d["relevant_isbns"]

        if not rel:
            continue

        for src in retrievers:
            ranked = d.get(src, [])

            if not ranked:
                continue

            for k in ks:

                topk = ranked[:k]

                # 맞춘 ISBN
                hit_isbns = set(topk) & set(rel)

                rows.append({
                    "query_id": qid,
                    "strategy": src,
                    "k": k,

                    # metric
                    "hit": hit_at_k(rel, ranked, k),
                    "recall": recall_at_k(rel, ranked, k),
                    "mrr": mrr_at_k(rel, ranked, k),
                    "ndcg": ndcg_at_k(rel, ranked, k),

                    # 추가 count 정보
                    "goldset_count": len(rel),
                    "retrieved_relevant_count": len(hit_isbns),

                    # 어떤 책 맞췄는지
                    "matched_isbns": list(hit_isbns),
                })

    return pd.DataFrame(rows)

retrieval_df = compute_retrieval_metrics(

    eval_data=eval_data,

    retrievers=strategies,

    ks=K_VALUES,

)

summary_df = (

    retrieval_df

    .groupby("strategy")

    .agg({

        "hit": "mean",

        "recall": "mean",

        "mrr": "mean",

        "ndcg": "mean",

        # count

        "retrieved_relevant_count": "sum",

        "goldset_count": "sum",

    })

    .reset_index()

)

display(summary_df)

,strategy,hit,recall,mrr,ndcg,retrieved_relevant_count,goldset_count
0,strategy0,0.952381,0.311418,0.742914,0.468449,155,580
1,strategy1,0.952381,0.273019,0.577372,0.397797,139,580
2,strategy2,0.952381,0.231555,0.685374,0.374039,118,580


In [13]:
# =====================================================
# 7. strategy 평균 성능
# =====================================================

summary = (
    retrieval_df
    .groupby(["strategy", "k"])[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
)

print("[strategy 평균 성능]")
print(summary.to_string())

[strategy 평균 성능]
                 hit  recall     mrr    ndcg
strategy  k                                 
strategy0 20  0.9524  0.3114  0.7429  0.4684
strategy1 20  0.9524  0.2730  0.5774  0.3978
strategy2 20  0.9524  0.2316  0.6854  0.3740


In [14]:
retrieval_df.to_csv(
    "/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_3_metric_results.csv",
    index=False,
    encoding="utf-8-sig"
)

In [1]:
import json
import pandas as pd
from pathlib import Path

# =====================================================
# 1. 파일 경로
# =====================================================

METRIC_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_3_metric_results.csv")
SCENARIO_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/scenario_data_after_anchor_rewrite.json")
SAVE_DIR = Path("/home/ubuntu/work/dahyeon/evaluation/dataset")

# =====================================================
# 2. 데이터 로드
# =====================================================

metric_df = pd.read_csv(METRIC_PATH)

with SCENARIO_PATH.open("r", encoding="utf-8") as f:
    scenario_data = json.load(f)

# =====================================================
# 3. query_type 추가
# =====================================================

query_type_map = {
    item["scenario_id"]: item["query_type"]
    for item in scenario_data
}

metric_df["query_type"] = metric_df["query_id"].map(query_type_map)

# =====================================================
# 4. 전체 평균 비교
# =====================================================

overall_result = (
    metric_df
    .groupby("strategy")[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
    .reset_index()
)

print("=== 전체 평균 성능 ===")
print(overall_result)

# =====================================================
# 5. query_type별 성능 비교
# =====================================================

type_result = (
    metric_df
    .groupby(["query_type", "strategy"])[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
    .reset_index()
)

print("\n=== Query Type별 성능 ===")
print(type_result)

# =====================================================
# 6. query_type별 strategy recall 비교
# =====================================================

type_recall_pivot = type_result.pivot(
    index="query_type",
    columns="strategy",
    values="recall"
)

print("\n=== Query Type별 Recall 비교 ===")
print(type_recall_pivot)

# =====================================================
# 7. query별 strategy recall 비교
# =====================================================

query_recall_pivot = metric_df.pivot_table(
    index="query_id",
    columns="strategy",
    values="recall"
).reset_index()

print("\n=== Query별 Recall 비교 ===")
print(query_recall_pivot)

# =====================================================
# 8. strategy간 recall 차이 계산
# =====================================================

# strategy_cols = [
#     c for c in query_recall_pivot.columns
#     if c != "query_id"
# ]

# # strategy0 기준 차이 계산
# if "strategy0" in strategy_cols:

#     for col in strategy_cols:

#         if col == "strategy0":
#             continue

#         query_recall_pivot[f"{col}_minus_strategy0"] = (
#             query_recall_pivot[col]
#             - query_recall_pivot["strategy0"]
#         )

# print("\n=== Strategy Recall 차이 ===")
# print(query_recall_pivot)

# =====================================================
# 9. 결과 저장
# =====================================================

# metric_df.to_csv(
#     SAVE_DIR / "chunk_metric_with_query_type.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# overall_result.to_csv(
#     SAVE_DIR / "chunk_overall_result.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# type_result.to_csv(
#     SAVE_DIR / "chunk_query_type_result.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# type_recall_pivot.to_csv(
#     SAVE_DIR / "chunk_query_type_recall_compare.csv",
#     encoding="utf-8-sig"
# )

query_recall_pivot.to_csv(
    SAVE_DIR / "chunk_4_query_recall_compare.csv",
    index=False,
    encoding="utf-8-sig"
)

# print("\n저장 완료")

=== 전체 평균 성능 ===
    strategy     hit  recall     mrr    ndcg
0  strategy0  0.9524  0.3114  0.7429  0.4684
1  strategy1  0.9524  0.2730  0.5774  0.3978
2  strategy2  0.9524  0.2316  0.6854  0.3740

=== Query Type별 성능 ===
            query_type   strategy     hit  recall     mrr    ndcg
0         anchor_based  strategy0  0.6667  0.2588  0.6667  0.3261
1         anchor_based  strategy1  1.0000  0.4410  0.4653  0.3777
2         anchor_based  strategy2  0.6667  0.2784  0.6667  0.3281
3   availability_first  strategy0  1.0000  0.2738  0.7778  0.6110
4   availability_first  strategy1  1.0000  0.2007  0.4365  0.4051
5   availability_first  strategy2  1.0000  0.2031  1.0000  0.5502
6      broad_ambiguous  strategy0  1.0000  0.2257  0.7778  0.5043
7      broad_ambiguous  strategy1  1.0000  0.2553  1.0000  0.6412
8      broad_ambiguous  strategy2  1.0000  0.1606  0.8333  0.4392
9      pure_mood_state  strategy0  1.0000  0.2819  0.7143  0.3183
10     pure_mood_state  strategy1  1.0000  0.1829  0.

In [2]:
# =====================================================
# 10. query별 최고 전략 찾기
# =====================================================

strategy_cols = [
    c for c in query_recall_pivot.columns
    if c != "query_id"
]

query_recall_pivot["best_strategy"] = (
    query_recall_pivot[strategy_cols]
    .idxmax(axis=1)
)

query_recall_pivot["best_recall"] = (
    query_recall_pivot[strategy_cols]
    .max(axis=1)
)

# query_type 붙이기
query_recall_pivot["query_type"] = (
    query_recall_pivot["query_id"]
    .map(query_type_map)
)

print("\n=== Query별 최고 전략 ===")
print(query_recall_pivot)


=== Query별 최고 전략 ===
strategy query_id  strategy0  strategy1  strategy2 best_strategy  best_recall  \
0             S01   0.176471   0.411765   0.235294     strategy1     0.411765   
1             S02   0.000000   0.111111   0.000000     strategy1     0.111111   
2             S03   0.600000   0.800000   0.600000     strategy1     0.800000   
3             S04   0.500000   0.500000   0.500000     strategy0     0.500000   
4             S05   0.363636   0.090909   0.181818     strategy0     0.363636   
5             S06   0.229167   0.250000   0.166667     strategy1     0.250000   
6             S07   0.750000   0.875000   0.750000     strategy1     0.875000   
7             S08   0.352941   0.058824   0.235294     strategy0     0.352941   
8             S09   0.357143   0.285714   0.142857     strategy0     0.357143   
9             S10   0.261905   0.285714   0.214286     strategy1     0.285714   
10            S11   0.375000   0.000000   0.125000     strategy0     0.375000   
11    

In [3]:
strategy0_win = query_recall_pivot[
    query_recall_pivot["best_strategy"] == "strategy0"
]

print(strategy0_win[
    ["query_id", "query_type", "strategy0", "strategy1", "strategy2"]
])

strategy query_id          query_type  strategy0  strategy1  strategy2
3             S04       topic_purpose   0.500000   0.500000   0.500000
4             S05       topic_purpose   0.363636   0.090909   0.181818
7             S08    topic_constraint   0.352941   0.058824   0.235294
8             S09    topic_constraint   0.357143   0.285714   0.142857
10            S11          pure_topic   0.375000   0.000000   0.125000
11            S12          pure_topic   0.229508   0.147541   0.196721
12            S13     pure_mood_state   0.258065   0.161290   0.161290
13            S14     pure_mood_state   0.187500   0.187500   0.062500
14            S15     pure_mood_state   0.400000   0.200000   0.200000
15            S16  availability_first   0.190476   0.142857   0.095238
16            S17  availability_first   0.384615   0.256410   0.282051
17            S18  availability_first   0.246377   0.202899   0.231884
20            S21     broad_ambiguous   0.305556   0.194444   0.138889


In [9]:
# =====================================================
# query text 추출
# =====================================================

query_text_map = {}

for item in scenario_data:

    qid = item["scenario_id"]

    query = None

    # rag_query.semantic_query 추출
    for turn in reversed(item.get("turns", [])):

        if "rag_query" in turn:

            query = (
                turn["rag_query"]
                .get("semantic_query")
            )

            break

    query_text_map[qid] = query

# =====================================================
# query 추가
# =====================================================

query_recall_pivot["query"] = (
    query_recall_pivot["query_id"]
    .map(query_text_map)
)

# =====================================================
# 출력
# =====================================================

print("\n=== Query 포함 Recall 비교 ===")

print(
    query_recall_pivot[
        [
            "query_id",
            "query",
            "strategy0",
            "strategy1",
            "strategy2"
        ]
    ]
)


=== Query 포함 Recall 비교 ===
strategy query_id                                              query  \
0             S01  실존주의적 주제를 탐구하며 철학적 성찰을 통해 독자에게 깊은 사색을 유도하는 인문학...   
1             S02           한국 경제의 문제점과 현실적인 해결 방안을 제시하는 경제/경영 분야의 책   
2             S03  복잡한 우주와 물리학 개념을 대중적인 서술 방식으로 설명하여 일반인도 쉽게 이해할 ...   
3             S04                      팀장이 된 중급 독자를 위한 실무 리더십·팀 관리 책   
4             S05                  노년기 삶을 주제로 한 가볍게 읽히는 한국 작가 인문 에세이   
5             S06                     인간관계가 힘들 때 위로와 공감이 되는 자기계발·에세이   
6             S07                    주식·ETF 기초부터 실전 위주로 쉽게 설명한 재테크 책   
7             S08                 역사적 인물의 삶과 업적을 중심으로 한 세계사 배경의 역사소설   
8             S09             공감되면서 힘이 되는 여성 서사 한국소설, 무겁지 않고 우울하지 않은   
9             S10                  나이 들면서 어떻게 살아야 할지 생각하게 하는 따뜻한 에세이   
10            S11                          슬프지 않고 가볍게 읽히는 청춘·성장 한국소설   
11            S12                                  한국 여성 시인의 현대시 작품집   
12            S13             스트레스 받